# Free Diagnostics — Pillar 4 + Headroom, all classes (CAMUS + BRISC)

Warm-starts the **lite U-Net** baseline for every class and runs the **free, CPU-only**
diagnostics before any GPU/RL training round:

- **Pillar 4** (`pillar4_report`) — is the prob_map usable (not overconfident), and is the
  baseline's error boundary-shaped (contour-fixable) or topology/interior-shaped (structurally
  capped, no contour action can fix it)?
- **Headroom** (`headroom_report`) — non-GT-privileged realistic headroom vs. the attention U-Net
  competitor's Dice on the same class (`attention_dice`), alongside the GT-privileged oracle ceiling
  (upper bound on the action space only — never an achievable target).

Nothing here trains an agent or touches a GPU beyond the one-time U-Net warm-start pass per
dataset. Run this **before** committing a Kaggle GPU round to any class — skip/deprioritise any
class that comes back `CAPPED` or `NO HEADROOM`.

**Attach in Datasets (right panel) before running:** `iteris-pkg`, `CAMUS`, `brisc2025` (or `BRISC`),
the lite-U-Net checkpoint dataset(s) containing `camus_lite_unet_best.pt` / `brisc_lite_unet_best.pt`.

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached. Add it in the right-hand panel → Datasets.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Imports + helper

`attention_dice` per class is the attention U-Net's own fully-supervised test Dice (the fixed
competitor) — used to compute a *realistic*, non-GT-privileged headroom estimate. Subtype BRISC
configs (glioma/meningioma/pituitary) don't have a dedicated attention-Dice number yet, so they
report the GT-privileged ceiling only (still useful as a CAPPED/not-CAPPED signal).

`run_class_diagnostics` never raises — a failure on one class is logged and skipped so the rest of
the notebook still completes.

In [ ]:
from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils import model_suffix
from iteris.warm_start import precompute_init_masks
from iteris.diagnostics import pillar4_report, headroom_report
import numpy as np
import pandas as pd

# Attention U-Net competitor test Dice, per class (see CONTEXT.md / SKILLS.md).
ATTN_DICE = {
    'CAMUS_LV_endo': 0.938,
    'CAMUS_LV_epi':  0.872,
    'CAMUS_LA':      0.896,
    'BRISC_tumor':   0.835,
    # subtype-specific attention Dice not yet measured -> None (ceiling-only verdict)
    'BRISC_glioma':     None,
    'BRISC_meningioma': None,
    'BRISC_pituitary':  None,
}

RESULTS = {}   # label -> dict(pillar4=..., headroom=..., n_train=..., n_val=..., n_test=...)
SAMPLES = {}   # label -> dict(train=..., val=..., test=...)  (kept in memory for later cells)


def run_class_diagnostics(config_path, dataset_root_names, label,
                           min_area_fraction_key='min_area_fraction'):
    """Warm-start the lite U-Net for one class/config and run Pillar 4 + headroom.
    Geometry (n_points/cont_sectors/disp_px/spline_smooth) is read off the TD3 block,
    which carries the full locked geometry for every class regardless of which agent
    will eventually be trained on it."""
    try:
        cfg_full = load_drl_class_config(str(PKG_ROOT / 'configs' / config_path))
        cfg = resolve_agent_config(cfg_full, 'TD3')
        baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

        data_root = None
        for name in dataset_root_names:
            cands = [p for p in Path('/kaggle/input').rglob(name) if p.is_dir()]
            if cands:
                data_root = str(cands[0])
                break
        if data_root is None:
            raise FileNotFoundError(f'none of {dataset_root_names} found under /kaggle/input')
        baseline_cfg['data_root'] = data_root

        ckpt_name = f"{baseline_cfg['dataset'].lower()}{model_suffix(baseline_cfg)}_best.pt"
        ckpt_cands = list(Path('/kaggle/input').rglob(ckpt_name))
        if not ckpt_cands:
            raise FileNotFoundError(f'baseline checkpoint {ckpt_name!r} not found under /kaggle/input')
        cfg['baseline_checkpoint'] = str(ckpt_cands[0])

        print(f"\n{'='*70}\n{label}\n{'='*70}")
        print(f'data_root  : {data_root}')
        print(f"checkpoint : {cfg['baseline_checkpoint']}")

        train_s, val_s, test_s = precompute_init_masks(
            baseline_cfg=baseline_cfg,
            baseline_checkpoint=cfg['baseline_checkpoint'],
            target_class=cfg['target_class'],
            min_area_fraction=cfg.get(min_area_fraction_key, 0.01),
            tumor_type_filter=cfg.get('tumor_type_filter'))
        print(f'samples — train {len(train_s)}  val {len(val_s)}  test {len(test_s)}')

        pillar4 = pillar4_report(val_s, label=label)
        headroom = headroom_report(
            val_s, n_points=cfg['n_points'], cont_sectors=cfg.get('cont_sectors', 12),
            disp_px=cfg['disp_px'], spline_smooth=cfg['spline_smooth'],
            n_samples=60, label=label, attention_dice=ATTN_DICE.get(label))

        SAMPLES[label] = dict(train=train_s, val=val_s, test=test_s)
        return dict(pillar4=pillar4, headroom=headroom,
                    n_train=len(train_s), n_val=len(val_s), n_test=len(test_s))
    except Exception as e:
        print(f'[FAILED] {label}: {e}')
        return None

## 2 · CAMUS — warm-start + free diagnostics (LV-endo / LV-epi / LA)

~5 min per class for the U-Net warm-start pass (cached internally per class, not reused across
classes — each `target_class` needs its own backbone pass). Diagnostics themselves are instant.

In [ ]:
CAMUS_CONFIGS = [
    ('camus_drl_c1.yaml', 'CAMUS_LV_endo'),
    ('camus_drl_c2.yaml', 'CAMUS_LV_epi'),
    ('camus_drl_c3.yaml', 'CAMUS_LA'),
]

for config_path, label in CAMUS_CONFIGS:
    RESULTS[label] = run_class_diagnostics(config_path, dataset_root_names=['CAMUS'], label=label)

## 3 · BRISC — warm-start + free diagnostics (generic tumour)

This is the class in the core run matrix (`PLAN.md` §4). ~8–15 min for the U-Net warm-start pass
(9 586 samples).

In [ ]:
RESULTS['BRISC_tumor'] = run_class_diagnostics(
    'brisc_drl_tumor.yaml', dataset_root_names=['brisc2025', 'BRISC'], label='BRISC_tumor')

## 4 · [OPTIONAL] BRISC tumour subtypes — glioma / meningioma / pituitary

Each subtype re-runs the full U-Net warm-start pass (another ~8–15 min each — 3× the cost of
§3). These are explicitly optional in `PLAN.md` ("if time permits"), so they're **off by default**.
Set `RUN_BRISC_SUBTYPES = True` to run them.

In [ ]:
RUN_BRISC_SUBTYPES = False   # set True to also warm-start + diagnose the 3 BRISC subtypes

BRISC_SUBTYPE_CONFIGS = [
    ('brisc_drl_glioma.yaml',     'BRISC_glioma'),
    ('brisc_drl_meningioma.yaml', 'BRISC_meningioma'),
    ('brisc_drl_pituitary.yaml',  'BRISC_pituitary'),
]

if RUN_BRISC_SUBTYPES:
    for config_path, label in BRISC_SUBTYPE_CONFIGS:
        RESULTS[label] = run_class_diagnostics(
            config_path, dataset_root_names=['brisc2025', 'BRISC'], label=label)
else:
    print('Skipped (RUN_BRISC_SUBTYPES = False). Flip the flag above to run them.')

## 5 · Summary table

One row per class. `headroom` and `oracle_ceiling` are GT-privileged (upper bound on the action
space only). `realistic_headroom` (= attention Dice − baseline Dice) is the honest, non-GT number —
read **that** column when deciding whether a class is worth a GPU training round. `pillar4_go`
combines the prob_map-informativeness and error-type checks into one go/no-go flag.

In [ ]:
rows = []
for label, r in RESULTS.items():
    if r is None:
        rows.append(dict(label=label, status='FAILED'))
        continue
    h, p4 = r['headroom'], r['pillar4']
    rows.append(dict(
        label=label, status='ok',
        n_val=r['n_val'],
        baseline_dice=h['baseline_init_dice'],
        contour_repr_ceiling=h['contour_repr_ceiling'],
        oracle_ceiling_GT_PRIVILEGED=h['oracle_greedy_ceiling'],
        attention_dice=h.get('attention_dice'),
        realistic_headroom=h.get('realistic_headroom_estimate'),
        boundary_frac=p4['error_type']['boundary_frac'],
        topology_frac=p4['error_type']['topology_frac'],
        interior_frac=p4['error_type']['interior_frac'],
        prob_map_uncertain_band_frac=p4['prob_map']['uncertain_band_frac'],
        pillar4_go=p4['go'],
    ))

summary = pd.DataFrame(rows).set_index('label')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)
display(summary)

out = '/kaggle/working/diagnostics_summary.csv'
summary.to_csv(out)
print(f'\nSaved to {out}')

print('\n--- Plain-language verdicts ---')
for label, r in RESULTS.items():
    if r is None:
        print(f'{label:20s}: FAILED — see error above')
        continue
    h, p4 = r['headroom'], r['pillar4']
    realistic = h.get('realistic_headroom_estimate')
    headroom_verdict = (
        'NO realistic headroom number (no attention_dice set for this class -- ceiling-only)'
        if realistic is None else
        'GOOD -- train it' if realistic > 0.02 else
        'MARGINAL -- small reachable gain' if realistic > 0.005 else
        'NO HEADROOM -- skip GPU training for this class')
    cap = p4['error_type']['topology_frac'] + p4['error_type']['interior_frac']
    shape_verdict = ('boundary-fixable' if p4['error_type']['boundary_frac'] > 0.7
                      else f"CAPPED ({cap:.0%} topology/interior -- contour cannot fix this)")
    print(f'{label:20s}: headroom={headroom_verdict:55s} | error-shape={shape_verdict}')